In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CBMSP10", con=connection2)

connection2.close()




In [2]:
base2

,motsusprogcod,motsusprogdes,motsushrsflg
0,01,ORDEN JEF.SERVICIO/DIRECCION,1
1,02,FALTA INJUSTIFICADA,0
2,03,DESCANSO MEDICO,1
3,04,CAMBIO DE TURNO,0
4,05,COMISION DE SERVICIO,1
5,06,LICENCIA,1
6,07,VACACIONES,1
7,08,ONOMASTICO,1
8,09,REPROGRAMACION,0
9,99,ERROR DE DIGITACION,0


In [3]:
a=base2[base2.duplicated(subset=["motsusprogcod"])]
a

,motsusprogcod,motsusprogdes,motsushrsflg


In [4]:
base2.columns

Index(['motsusprogcod', 'motsusprogdes', 'motsushrsflg'], dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_cbmsp10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbmsp10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_cbmsp10 
ALTER COLUMN motsusprogcod TYPE CHARACTER (2), 
ALTER COLUMN motsusprogdes TYPE CHARACTER (40), 
ALTER COLUMN motsushrsflg TYPE CHARACTER (15); 


UPDATE cbmsp10 
SET 


motsusprogcod             =t.motsusprogcod ,
motsusprogdes             =t.motsusprogdes ,
motsushrsflg           =t.motsushrsflg 



FROM tmp_cbmsp10 t 
WHERE cbmsp10.motsusprogcod = t.motsusprogcod and cbmsp10.motsusprogcod IS NOT NULL and t.motsusprogcod IS NOT NULL ;

INSERT INTO cbmsp10 
(motsusprogcod, motsusprogdes, motsushrsflg) 

SELECT 
motsusprogcod, motsusprogdes, motsushrsflg

FROM tmp_cbmsp10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cbmsp10 
    WHERE cbmsp10.motsusprogcod = tmp_cbmsp10.motsusprogcod and cbmsp10.motsusprogcod IS NOT NULL and tmp_cbmsp10.motsusprogcod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cbmsp10;
DELETE FROM cbmsp10 WHERE motsusprogcod ='';
"""


c2= text(query2)
connection3.execute(c2)
base2 = pd.read_sql_query(f"SELECT * FROM cbmsp10", con=connection3)

connection3.close()


In [6]:
base2.columns

Index(['motsusprogcod', 'motsusprogdes', 'motsushrsflg'], dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cbmsp10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""

ALTER TABLE tmp_cbmsp10 
ALTER COLUMN motsusprogcod TYPE CHARACTER (2), 
ALTER COLUMN motsusprogdes TYPE CHARACTER (40), 
ALTER COLUMN motsushrsflg TYPE CHARACTER (15); 


UPDATE dim_motsuspro 
SET 

cod_mot       = t.motsusprogcod,
des_mot       = t.motsusprogdes


FROM tmp_cbmsp10 t 
WHERE dim_motsuspro.cod_mot = t.motsusprogcod AND dim_motsuspro.cod_mot  IS NOT NULL;


INSERT INTO dim_motsuspro (cod_mot, des_mot) 
SELECT motsusprogcod, motsusprogdes    

FROM tmp_cbmsp10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_motsuspro 
    WHERE dim_motsuspro.cod_mot = tmp_cbmsp10.motsusprogcod
);

DROP TABLE tmp_cbmsp10;

"""



c1= text(query)
connection4.execute(c1)
connection4.close()